# 大连校园共享空间的纯运动预测

**设置**　私有 Notebook。加速器 CPU，Internet 开启。

**输入**　挂载私有输入文件。作者轨迹与比例尺从固定版本读取。

**打包时状态**　窗口清单与权重来源已核实，尚未收到这项任务的 Kaggle 结果归档。

冻结 PINNS 的四组 M0 权重，每片读作者提供的轨迹与比例尺，以原视频呈现时间比较网络与末两点匀速。大连同一校园机位、11 片、3,066 窗、879 个片内行人身份。原视频时钟约 23.976 Hz。

**输出**　`/kaggle/working/dut_shared_space_motion.zip`。

In [ ]:
"""Private Kaggle launcher. Scientific scripts run in separate Python processes."""
import base64
import hashlib
import io
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zipfile
import zlib

EXPECTED = {
    'shortpose_heldout.zip':'507c0a5decf79a4a48b982987e89c37b8c99e456f5c7f11cc3d49f34bd5741ec',
    'pose.npz':'b493a47536664ddb4fd66603c1bbf6b2c2e33f553520382027b3ae96b53447d0',
    'reranked_modes.zip':'b24ac2bb1ad3d6742be2cbe16b927e42ef9a1a54a17ce2c08be3e92fa286b871',
}
BUNDLE_NAME = 'pinns_private_inputs.bin'
BUNDLE_SHA = 'b31928314a7e8788b7a013655a58fd6b96ae0d9cac7b25b274690d5a0682695d'

def digest(data):
    return hashlib.sha256(data).hexdigest()

def input_view(names, mount, staging):
    """Prefer the natural mounted originals. Otherwise open the supplied byte-preserving bundle."""
    if not names:
        return mount
    direct = {name: sorted(mount.rglob(name)) for name in names}
    if all(len(paths) == 1 and digest(paths[0].read_bytes()) == EXPECTED[key]
           for key, paths in direct.items()):
        return mount
    if names in (['shortpose_heldout.zip'], ['pose.npz']):
        five_seed_sha = '0bf2745ab2d5ddcf5f5fbb5d330e6c6cc800625cec301f4c7105d031bdb49d2d'
        if any(p.stat().st_size == 42019120 and digest(p.read_bytes()) == five_seed_sha
               for p in mount.rglob('shortpose_heldout_five_seeds.zip')):
            return mount
    bundles = sorted(mount.rglob(BUNDLE_NAME))
    if names == ['shortpose_heldout.zip'] and not bundles and any(mount.rglob('M0.pt')):
        # The frozen DUT entrypoints check each site's exact weight hash themselves.
        return mount
    if not bundles or any(digest(p.read_bytes()) != BUNDLE_SHA for p in bundles):
        raise FileNotFoundError(f'请用 Add Input 挂载本包中的 {BUNDLE_NAME} 或完整的原始输入 {names}')
    staging.mkdir()
    with zipfile.ZipFile(bundles[0]) as source:
        for name in names:
            if name == 'pose.npz':
                with zipfile.ZipFile(io.BytesIO(source.read('shortpose_heldout.zip'))) as nested:
                    raw = nested.read(name)
            else:
                raw = source.read(name)
            if digest(raw) != EXPECTED[name]:
                raise ValueError('输入校验失败 ' + name)
            (staging/name).write_bytes(raw)
    return staging

# The wrapper aliases only the launcher's literal Path('/kaggle/input') when a
# byte-preserving attachment was staged. Imported project modules use normal pathlib.
CHILD = '''import builtins, os, pathlib, runpy, sys, types
source = pathlib.Path(sys.argv[1])
root = pathlib.Path(sys.argv[2])
sys.argv = [str(source)]
if root == pathlib.Path('/kaggle/input'):
    runpy.run_path(str(source), run_name='__main__')
else:
    proxy = types.ModuleType('pathlib')
    def mounted_path(*parts):
        path = pathlib.Path(*parts)
        return root if path == pathlib.Path('/kaggle/input') else path
    proxy.Path = mounted_path
    normal_import = builtins.__import__
    def launcher_import(name, globals=None, locals=None, fromlist=(), level=0):
        if name == 'pathlib' and tuple(fromlist or ()) == ('Path',) and level == 0:
            return proxy
        return normal_import(name, globals, locals, fromlist, level)
    context = {'__name__':'__main__', '__file__':str(source),
               '__builtins__':dict(vars(builtins), __import__=launcher_import)}
    exec(compile(source.read_bytes(), str(source), 'exec'), context)
'''

def run_frozen(filename, expected, packed, required, output, detector_device=None):
    mount = Path('/kaggle/input')
    working = Path('/kaggle/working')
    if not (mount.is_dir() and working.is_dir()):
        raise RuntimeError('请在 Kaggle 私有 Notebook 中运行此单元')
    source = zlib.decompress(base64.b64decode(packed, validate=True))
    if digest(source) != expected:
        raise ValueError('冻结运行入口校验失败 ' + filename)
    with tempfile.TemporaryDirectory(prefix='private_forecasts_') as name:
        tmp = Path(name)
        launcher = tmp/filename
        launcher.write_bytes(source)
        view = input_view(required, mount, tmp/'inputs')
        print('入口', filename, 'SHA-256', expected, flush=True)
        env = {**os.environ, 'PYTHONDONTWRITEBYTECODE':'1'}
        if detector_device is not None:
            if detector_device not in ('auto', 'cpu', 'cuda'):
                raise ValueError('检测器后端须为 auto、cpu 或 cuda')
            env['DUT_DETECTOR_DEVICE'] = detector_device
        subprocess.run([sys.executable, '-u', '-c', CHILD, str(launcher), str(view)],
                       env=env, check=True)
    result = working/output
    with zipfile.ZipFile(result) as z:
        if z.testzip() is not None:
            raise ValueError('结果归档 CRC 校验失败')
    print('结果', result, result.stat().st_size, 'bytes；SHA-256', digest(result.read_bytes()), flush=True)


## 第 1 步　冻结 PINNS 的四组 M0 权重，每片读作者提供的轨迹与比例尺，以原视频呈现时间比较网络与末两点匀速。大连同一校园机位、11 片、3,066 窗、879 个片内行人身份。原视频时钟约 23.976 Hz。

冻结入口 SHA-256　`6970ac111ed3a69c7003ecf9c369501d72eb23587d2e128c78dfd1f27d2abc9f`。

In [ ]:
run_frozen(
    'kaggle_dut_shared_space_motion.py', '...'
    ),
    ['shortpose_heldout.zip'], 'dut_shared_space_motion.zip'
)
